# KINNEWS - GPU experiments

Runs the BiLSTM experiments (E7-E9) and the Transformer fine-tuning (T1-T5).
Made for Kaggle: turn on a GPU and Internet in the session options, then use
**Save Version -> Save & Run All (Commit)** so it runs in the background (~2-2.5 h).
Everything ends up in the notebook's Output tab: `results/`, `results.zip` and `final_model/`.
Works on Colab too.

In [1]:
import os
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
REPO_DIR = f"{WORK}/repo"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Samkwizera/NLP-application.git {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements-train.txt
!python scripts/prepare_data.py
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU!")

/kaggle/working/repo
{
  "official_train_rows": 17014,
  "official_test_rows": 4254,
  "train_conflicting_label_articles_dropped": 160,
  "train_test_overlap_dropped": 3,
  "train_rows": 6344,
  "val_rows": 1120,
  "test_rows": 3464,
  "train_original_rows": 17014
}
GPU: Tesla T4


## BiLSTM (E7-E9)
- E7 random embeddings, max pooling
- E8 Word2Vec embeddings, max pooling
- E9 Word2Vec embeddings, attention pooling

In [2]:
!python scripts/train_bilstm.py --run-name e7_bilstm_random_max --pooling max
!python scripts/train_bilstm.py --run-name e8_bilstm_w2v_max --pooling max --w2v
!python scripts/train_bilstm.py --run-name e9_bilstm_w2v_attention --pooling attention --w2v

epoch  1  train loss 2.1179  val macro-F1 0.1394
epoch  2  train loss 1.7458  val macro-F1 0.2867
epoch  3  train loss 1.3629  val macro-F1 0.3492
epoch  4  train loss 1.1156  val macro-F1 0.3535
epoch  5  train loss 0.9663  val macro-F1 0.3789
epoch  6  train loss 0.8747  val macro-F1 0.3956
epoch  7  train loss 0.7855  val macro-F1 0.4238
epoch  8  train loss 0.7062  val macro-F1 0.4504
epoch  9  train loss 0.6504  val macro-F1 0.4662
epoch 10  train loss 0.5816  val macro-F1 0.4813
epoch 11  train loss 0.5221  val macro-F1 0.4818
epoch 12  train loss 0.4853  val macro-F1 0.4930
epoch 13  train loss 0.4323  val macro-F1 0.4640
epoch 14  train loss 0.3915  val macro-F1 0.5090
epoch 15  train loss 0.3454  val macro-F1 0.5056
e7_bilstm_random_max                     val macro-F1 0.5090 | test acc 0.6094 macro-F1 0.3833
Word2Vec coverage: 50000/50000 vocabulary words
epoch  1  train loss 1.5801  val macro-F1 0.3239
epoch  2  train loss 1.0074  val macro-F1 0.3883
epoch  3  train loss 0.8

## Transformers (T1-T3)
- T1 `xlm-roberta-base`, general multilingual
- T2 `Davlan/afro-xlmr-base`, XLM-R with extra pretraining on African languages incl. Kinyarwanda
- T3 `castorini/afriberta_base`, pretrained from scratch on African languages incl. Kinyarwanda

Checkpoints are deleted after each run, only the scores and predictions are kept, to save disk.

In [3]:
!python scripts/train_transformer.py --model FacebookAI/xlm-roberta-base --run-name t1_xlmr_base
!rm -rf models/t1_xlmr_base
!python scripts/train_transformer.py --model Davlan/afro-xlmr-base --run-name t2_afroxlmr_base
!rm -rf models/t2_afroxlmr_base
!python scripts/train_transformer.py --model castorini/afriberta_base --run-name t3_afriberta_base
!rm -rf models/t3_afriberta_base

config.json: 100%|█████████████████████████████| 615/615 [00:00<00:00, 2.48MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 128kB/s]
sentencepiece.bpe.model: 100%|██████████████| 5.07M/5.07M [00:00<00:00, 194MB/s]
tokenizer.json: 100%|███████████████████████| 9.10M/9.10M [00:00<00:00, 112MB/s]

model.safetensors: downloading bytes: ███▍                  |  177MB, 12.9MB/s  
model.safetensors: downloading bytes: ███████               |  358MB, 28.0MB/s  
model.safetensors: downloading bytes: ████████▉             |  456MB, 36.9MB/s  
model.safetensors: downloading bytes: █████████████         |  662MB, 54.6MB/s  
model.safetensors: downloading bytes: ██████████████▌       |  740MB, 62.4MB/s  
model.safetensors: downloading bytes: ██████████████████████|  745MB, 65.1MB/s  
model.safetensors: reconstructing file: 100%|██████| 1.12GB / 1.12GB, 99.8MB/s  
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2452.02it/s]
[transformers] XLMRobertaFo

## Picking the best model on validation
The best of T1-T3 is chosen by **val** macro-F1 only. The test set is not used for any decision.

In [4]:
import json
MODELS = {"t1_xlmr_base": "FacebookAI/xlm-roberta-base",
          "t2_afroxlmr_base": "Davlan/afro-xlmr-base",
          "t3_afriberta_base": "castorini/afriberta_base"}
val = {run: json.load(open(f"results/{run}.json"))["val"]["macro_f1"] for run in MODELS}
for run, score in val.items():
    print(f"{run:20s} val macro-F1 {score:.4f}")
BEST = MODELS[max(val, key=val.get)]
print("BEST =", BEST)

t1_xlmr_base         val macro-F1 0.4404
t2_afroxlmr_base     val macro-F1 0.5250
t3_afriberta_base    val macro-F1 0.6053
BEST = castorini/afriberta_base


## T4 and T5
- T4: best model with 512 tokens instead of 256 (less of each article is cut off)
- T5: same as T4 plus balanced class weights in the loss, to help the rare topics

In [5]:
!python scripts/train_transformer.py --model {BEST} --run-name t4_best_maxlen512 --max-length 512 --batch-size 8 --save-model
!rm -rf models/t4_best_maxlen512/checkpoint-*
!python scripts/train_transformer.py --model {BEST} --run-name t5_best_maxlen512_weighted --max-length 512 --batch-size 8 --class-weights --save-model
!rm -rf models/t5_best_maxlen512_weighted/checkpoint-*

Loading weights: 100%|█████████████████████| 133/133 [00:00<00:00, 14886.91it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  0%|             

## Summary and final model (picked on val)

In [6]:
import glob, shutil
rows = [json.load(open(f)) for f in sorted(glob.glob("results/*.json"))]
for d in rows:
    print(f"{d['name']:42s} val mF1 {d['val']['macro_f1']:.4f}  test acc {d['test']['accuracy']:.4f}  test mF1 {d['test']['macro_f1']:.4f}")

final_run = max(["t4_best_maxlen512", "t5_best_maxlen512_weighted"],
                key=lambda r: json.load(open(f"results/{r}.json"))["val"]["macro_f1"])
print("final model:", final_run)
shutil.copytree(f"models/{final_run}/final", f"{WORK}/final_model", dirs_exist_ok=True)
shutil.make_archive(f"{WORK}/results", "zip", "results")

e1_tfidf_word_logreg                       val mF1 0.5827  test acc 0.6966  test mF1 0.4261
e2_tfidf_word_nb                           val mF1 0.5626  test acc 0.7012  test mF1 0.4259
e2_tfidf_word_svm                          val mF1 0.6647  test acc 0.7182  test mF1 0.4784
e3_tfidf_word_logreg_no_elision_split      val mF1 0.5571  test acc 0.6836  test mF1 0.4143
e4_tfidf_char_logreg                       val mF1 0.6299  test acc 0.7047  test mF1 0.4511
e5_tfidf_word_logreg_balanced              val mF1 0.6823  test acc 0.7162  test mF1 0.4803
e6_tfidf_word_logreg_with_duplicates       val mF1 0.5856  test acc 0.6902  test mF1 0.4251
e7_bilstm_random_max                       val mF1 0.5090  test acc 0.6094  test mF1 0.3833
e8_bilstm_w2v_max                          val mF1 0.6171  test acc 0.6218  test mF1 0.4611
e9_bilstm_w2v_attention                    val mF1 0.6331  test acc 0.7168  test mF1 0.5348
t1_xlmr_base                               val mF1 0.4404  test acc 0.6989  test

'/kaggle/working/results.zip'

## Optional: upload the final model to the Hugging Face Hub
Only runs if a Kaggle secret called `HF_TOKEN` (a Write token) is attached under Add-ons -> Secrets.
Otherwise it is skipped and the model can be uploaded later from `final_model/`.

In [7]:
HUB_ID = "Samkwizera/kinnews-topic-classifier"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
if token:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    AutoModelForSequenceClassification.from_pretrained(f"{WORK}/final_model").push_to_hub(HUB_ID, token=token)
    AutoTokenizer.from_pretrained(f"{WORK}/final_model").push_to_hub(HUB_ID, token=token)
    print("uploaded to https://huggingface.co/" + HUB_ID)
else:
    print("no HF_TOKEN, skipping upload")

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

uploaded to https://huggingface.co/Samkwizera/kinnews-topic-classifier
